# Inspect remote SimpleFold results locally

Select the **Python (SimpleFold)** kernel. This notebook reads downloaded PDB
files only: it does not import folding models, download weights, or sample.
Place remote run directories under `artifacts/remote-runs/`; expected PDB paths
are `artifacts/remote-runs/JOB_ID/predictions_simplefold_100M/*.pdb`.
See [`../docs/hpc3y.md`](../docs/hpc3y.md) for remote job and transfer instructions.

The toy runs disable pLDDT. Their B-factors are placeholders; the viewer colors
by sequence position, and the summary checks sequence and coordinate content.


In [ ]:
from pathlib import Path
import gemmi
import numpy as np
import pandas as pd
import py3Dmol
from IPython.display import display

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src/simplefold").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Launch this notebook from the ml-simplefold checkout.")
REMOTE_RUNS = REPO_ROOT / "artifacts/remote-runs"


In [ ]:
pdb_paths = sorted(
    REMOTE_RUNS.glob("*/predictions_simplefold_100M/*.pdb"),
    key=lambda path: path.stat().st_mtime,
)
if not pdb_paths:
    raise FileNotFoundError(
        f"No remote PDBs under {REMOTE_RUNS}. Download a completed run first; "
        "see docs/hpc3y.md."
    )
for path in pdb_paths:
    print(path.relative_to(REPO_ROOT))

# Choose a specific file here if desired; default to the latest downloaded PDB.
pdb_path = pdb_paths[-1]


In [ ]:
def summarize_pdb(path):
    structure = gemmi.read_structure(str(path))
    rows = []
    for model in structure:
        for chain in model:
            residues = list(chain)
            atoms = [atom for residue in residues for atom in residue]
            coordinates = np.array(
                [[atom.pos.x, atom.pos.y, atom.pos.z] for atom in atoms],
                dtype=float,
            ).reshape(-1, 3)
            sequence = "".join(
                gemmi.find_tabulated_residue(residue.name).one_letter_code
                for residue in residues
                if gemmi.find_tabulated_residue(residue.name).is_amino_acid()
            )
            rows.append({
                "file": str(path.relative_to(REPO_ROOT)),
                "model": model.num,
                "chain": chain.name,
                "sequence": sequence,
                "residues": len(sequence),
                "atoms": len(atoms),
                "CA_atoms": sum(atom.name == "CA" for atom in atoms),
                "finite_coordinates": bool(np.isfinite(coordinates).all()),
            })
    return rows

summary = pd.DataFrame([
    row for path in pdb_paths for row in summarize_pdb(path)
])
display(summary)


In [ ]:
print(f"Viewing {pdb_path.relative_to(REPO_ROOT)}")
view = py3Dmol.view(width=800, height=500)
view.addModel(pdb_path.read_text(), "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()
